# WP LLaVA-1.0 — Notebook 1 : Génération

**Objectif** : projeter CLS MAE → espace CLIP ViT-L/14 @ 224px (= visual encoder de LLaVA 1.0).

**Stratégie d'augmentation** : pour chaque image, générer N_MASKS=100 CLS MAE masqués (seeds 0..99),
puis créer K=50 représentations en moyennant GROUP_SIZE=5 tirages aléatoires parmi les 100.

**Stockage optimisé** :
- `mae_avg`  : (N×50, 1024) — représentations MAE moyennées, brutes
- `cls_clip` : (N, 1024)    — une seule fois par image, L2-normalisé
- `img_idx`  : (N×50,)      — index de l'image source
- `labels`   : (N,)

| Config | Images | Paires train | Masquage |
|--------|--------|-------------|----------|
| wp3_5 (pote) | ~117k (ImageNet-100) | 585k (K=5) | 75% natif MAE |
| ce notebook  | ~117k (ImageNet-100) | 5.85M (K=50, avg de 5/100) | 75% natif MAE |

In [ ]:
import os
import torch
import torch.nn.functional as F
from transformers import ViTImageProcessor, ViTMAEModel, CLIPModel, CLIPProcessor
from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from tqdm import tqdm

DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'
N_MASKS    = 100
K_GROUPS   = 50
GROUP_SIZE = 5

print(f'Device : {DEVICE}')
print(f'Paires générées par image : {K_GROUPS} (moyenne de {GROUP_SIZE} parmi {N_MASKS} masquages)')

## 1.1 Chargement MAE + CLIP ViT-L/14 @ 224px

On utilise `openai/clip-vit-large-patch14` (224px) — le visual encoder de LLaVA 1.0,
à distinguer de `openai/clip-vit-large-patch14-336` utilisé dans LLaVA 1.5.
Les deux modèles sont à la même résolution MAE (224px), ce qui rend la tâche de projection symétrique.

In [ ]:
MAE_PATH  = './vit-mae-large'                if os.path.exists('./vit-mae-large')                else 'facebook/vit-mae-large'
CLIP_PATH = './clip-vit-large-patch14'       if os.path.exists('./clip-vit-large-patch14')       else 'openai/clip-vit-large-patch14'

# MAE — résolution 224px
mae_processor = ViTImageProcessor.from_pretrained(MAE_PATH)
mae_encoder   = ViTMAEModel.from_pretrained(MAE_PATH).to(DEVICE)
mae_encoder.eval()
print(f'MAE mask_ratio natif : {mae_encoder.config.mask_ratio}')
print(f'→ {int(196 * (1 - mae_encoder.config.mask_ratio))} patches visibles sur 196')

# CLIP ViT-L/14 @ 224px — LLaVA 1.0
clip_model     = CLIPModel.from_pretrained(CLIP_PATH).to(DEVICE)
clip_processor = CLIPProcessor.from_pretrained(CLIP_PATH)
clip_model.eval()
print(f'CLIP image_size : {clip_model.config.vision_config.image_size}px')  # doit afficher 224
print(f'CLIP hidden_size : {clip_model.config.vision_config.hidden_size}')   # 1024 pour ViT-L

## 1.2 Téléchargement CLIP si absent

In [ ]:
# Exécuter seulement si le dossier n'existe pas encore
if not os.path.exists('./clip-vit-large-patch14'):
    from huggingface_hub import snapshot_download
    snapshot_download(
        repo_id='openai/clip-vit-large-patch14',
        local_dir='./clip-vit-large-patch14'
    )
    print('CLIP téléchargé.')
else:
    print('CLIP déjà présent.')

## 1.3 Chargement dataset ImageNet-100

In [ ]:
DATASET_DIR = './imagenet100-hf'
if not os.path.exists(f'{DATASET_DIR}/data'):
    from huggingface_hub import snapshot_download
    snapshot_download(repo_id='ilee0022/ImageNet100', repo_type='dataset', local_dir=DATASET_DIR)

# Résolution unique 224px — MAE et CLIP sont à la même résolution
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
])

class ImageNet100Dataset(Dataset):
    def __init__(self, split, transform=None):
        pattern = f'{DATASET_DIR}/data/{split}-*.parquet'
        self.ds = load_dataset('parquet', data_files={split: pattern})[split]
        self.transform = transform
    def __len__(self): return len(self.ds)
    def __getitem__(self, idx):
        img   = self.ds[idx]['image'].convert('RGB')
        label = self.ds[idx]['label']
        if self.transform: img = self.transform(img)
        return img, label

dataset_train = ImageNet100Dataset('train',      transform=transform)
dataset_val   = ImageNet100Dataset('validation', transform=transform)
print(f'Train : {len(dataset_train)} images → {len(dataset_train) * K_GROUPS:,} paires')
print(f'Val   : {len(dataset_val)} images → {len(dataset_val) * K_GROUPS:,} paires')

## 1.4 Génération des paires (CLS MAE moyen, CLS CLIP)

- **CLIP** : encodé une seule fois par image via `clip_model.vision_model` + `visual_projection` → vecteur L2-normalisé de dim 1024
- **MAE** : encodé N_MASKS=100 fois avec masques différents, puis K_GROUPS=50 moyennes de GROUP_SIZE=5
- Pas d'interpolation de résolution : MAE et CLIP travaillent tous les deux à 224px

In [ ]:
def generate_pairs(dataset, batch_size=32, desc=''):
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False,
                        num_workers=4, pin_memory=(DEVICE == 'cuda'))

    all_mae_avg  = []
    all_cls_clip = []
    all_img_idx  = []
    all_labels   = []
    global_idx   = 0

    for images, labels in tqdm(loader, desc=desc):
        B = images.shape[0]

        # --- CLS CLIP @ 224px (une seule fois par image) ---
        clip_in = clip_processor(images=list(images), return_tensors='pt', do_rescale=False)
        clip_in = {k: v.to(DEVICE) for k, v in clip_in.items()}
        with torch.no_grad():
            vision_out  = clip_model.vision_model(**clip_in)
            cls_clip_batch = clip_model.visual_projection(vision_out.pooler_output)  # (B, 1024)
        cls_clip_batch = F.normalize(cls_clip_batch.cpu().float(), dim=-1)

        # --- N_MASKS CLS MAE masqués @ 224px ---
        mae_in = mae_processor(images=list(images), return_tensors='pt', do_rescale=False)
        mae_in = {k: v.to(DEVICE) for k, v in mae_in.items()}

        cls_pool = []  # liste de (B, 1024)
        for seed in range(N_MASKS):
            gen   = torch.Generator().manual_seed(seed)
            noise = torch.rand(B, 196, generator=gen).to(DEVICE)
            with torch.no_grad():
                out = mae_encoder(**mae_in, noise=noise)
            cls_pool.append(out.last_hidden_state[:, 0].cpu().float())  # CLS token

        cls_pool = torch.stack(cls_pool)  # (N_MASKS, B, 1024)

        # --- K_GROUPS moyennes de GROUP_SIZE masquages ---
        mae_avg_batch = []
        for _ in range(K_GROUPS):
            chosen = torch.randperm(N_MASKS)[:GROUP_SIZE]
            avg    = cls_pool[chosen].mean(dim=0)  # (B, 1024)
            mae_avg_batch.append(avg)

        mae_avg_batch = torch.cat(mae_avg_batch, dim=0)  # (K_GROUPS*B, 1024)
        img_idx = torch.arange(global_idx, global_idx + B).repeat(K_GROUPS)

        all_mae_avg.append(mae_avg_batch)
        all_cls_clip.append(cls_clip_batch)
        all_img_idx.append(img_idx)
        all_labels.append(labels)
        global_idx += B

    return {
        'mae_avg':  torch.cat(all_mae_avg),
        'cls_clip': torch.cat(all_cls_clip),
        'img_idx':  torch.cat(all_img_idx),
        'labels':   torch.cat(all_labels),
    }

In [ ]:
data_train = generate_pairs(dataset_train, batch_size=32, desc='Train')
data_val   = generate_pairs(dataset_val,   batch_size=32, desc='Val')

torch.save(data_train, 'llava10_pairs_train.pt')
torch.save(data_val,   'llava10_pairs_val.pt')
print('Sauvegarde OK')
print(f'  mae_avg  : {data_train["mae_avg"].shape}')   # (N*K, 1024)
print(f'  cls_clip : {data_train["cls_clip"].shape}')  # (N, 1024)
print(f'  img_idx  : {data_train["img_idx"].shape}')   # (N*K,)

In [ ]:
# Stats de normalisation — sauvegardées mais NON appliquées ici
# Le premier LayerNorm du ResidualBlock s'en charge à l'entraînement
norm_mean = data_train['mae_avg'].mean(dim=0)
norm_std  = data_train['mae_avg'].std(dim=0).clamp(min=1e-6)
torch.save({'mean': norm_mean, 'std': norm_std}, 'llava10_norm_stats.pt')
print('Stats de normalisation sauvegardées (pour référence / inférence externe)')

# Vérification discriminabilité inter-images
idx5  = [i * K_GROUPS for i in range(5)]
vecs  = F.normalize(data_val['mae_avg'][idx5], dim=-1)
sim   = (vecs @ vecs.T).numpy()
print('\nSimilarités cosinus mae_avg inter-images :')
print(sim.round(4))

# Vérification intra-image
vecs2 = F.normalize(data_val['mae_avg'][:5], dim=-1)
sim2  = (vecs2 @ vecs2.T).numpy()
print('\nSimilarités cosinus mae_avg intra-image (5 groupes, image 0) :')
print(sim2.round(4))

del mae_encoder, clip_model
torch.cuda.empty_cache()
print('VRAM libérée')

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from torch.optim import AdamW
import matplotlib.pyplot as plt

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {DEVICE}')

## 2.1 Chargement des paires

Les inputs MAE sont chargés **bruts** (non normalisés externement).
Le premier `LayerNorm` de chaque `ResidualBlock` assure la normalisation à l'entrée de chaque bloc.

In [ ]:
data_train = torch.load('llava10_pairs_train.pt')
data_val   = torch.load('llava10_pairs_val.pt')

# Reconstruire les paires (mae_avg, cls_clip) en expandant img_idx
mae_train  = data_train['mae_avg']                                                      # (N*K, 1024)
clip_train = data_train['cls_clip'][data_train['img_idx'] % len(data_train['cls_clip'])]  # (N*K, 1024)
mae_val    = data_val['mae_avg']
clip_val   = data_val['cls_clip'][data_val['img_idx'] % len(data_val['cls_clip'])]

print(f'Train : {len(mae_train):,} paires | Val : {len(mae_val):,} paires')
print(f'mae_train  — mean: {mae_train.mean():.4f}, std: {mae_train.std():.4f}')
print(f'clip_train — mean: {clip_train.mean():.4f}, std: {clip_train.std():.4f}')

BATCH_SIZE   = 8192
train_loader = DataLoader(TensorDataset(mae_train, clip_train),
                          batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(TensorDataset(mae_val, clip_val),
                          batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

## 2.2 Architecture MLP résiduel profond

Blocs résiduels pré-norm : `LayerNorm → Linear → GELU → Dropout → Linear + skip`.
Le skip connection stabilise le gradient sur 6 blocs (analogue aux résidus de ResNet / ViT).
Pas de tête de projection finale : la sortie du dernier bloc est directement L2-normalisée,
ce qui préserve la dimension 1024 cohérente avec l'espace CLIP ViT-L.

In [ ]:
class ResidualBlock(nn.Module):
    """Bloc résiduel pré-norm : LayerNorm -> Linear -> GELU -> Dropout -> Linear + skip."""
    def __init__(self, dim=1024, dropout=0.3):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.lin1  = nn.Linear(dim, dim)
        self.act   = nn.GELU()
        self.drop  = nn.Dropout(dropout)
        self.norm2 = nn.LayerNorm(dim)
        self.lin2  = nn.Linear(dim, dim)

    def forward(self, x):
        h = self.lin1(self.norm1(x))
        h = self.drop(self.act(h))
        h = self.lin2(self.norm2(h))
        return x + h


class ProjectionMLP(nn.Module):
    def __init__(self, dim=1024, n_blocks=6, dropout=0.3):
        super().__init__()
        self.blocks = nn.ModuleList([
            ResidualBlock(dim, dropout) for _ in range(n_blocks)
        ])

    def forward(self, x):
        for block in self.blocks:
            x = block(x)
        return F.normalize(x, dim=-1)


f_theta = ProjectionMLP().to(DEVICE)
print(f'Paramètres : {sum(p.numel() for p in f_theta.parameters()):,}')

## 2.3 Entraînement

- **Loss** : cosinus (1 − cos_sim), minimisée → alignement géométrique direct des espaces
- **Scheduler** : `ReduceLROnPlateau` sur la distance euclidienne val (plus sensible que la loss cosinus en fin d'entraînement)
- **Early stopping** : patience=20 sur la distance euclidienne val
- **Checkpoint** : meilleur modèle sauvegardé selon la distance euclidienne val minimale

In [ ]:
N_EPOCHS     = 200
LR           = 1e-4
WEIGHT_DECAY = 1e-4
PATIENCE     = 20

optimizer = AdamW(f_theta.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5, verbose=True
)
history = {'train_loss': [], 'val_loss': [], 'val_cos': [], 'val_euc': []}
best_euc, best_epoch, patience_counter = float('inf'), 0, 0


def cosine_loss(a, b):
    return (1 - F.cosine_similarity(a, b)).mean()


def eval_epoch(loader):
    f_theta.eval()
    tl, tc, te, n = 0., 0., 0., 0
    with torch.no_grad():
        for z, c in loader:
            z, c = z.to(DEVICE), c.to(DEVICE)
            p = f_theta(z)
            tl += cosine_loss(p, c).item() * z.shape[0]
            tc += F.cosine_similarity(p, c).mean().item() * z.shape[0]
            te += torch.norm(p - c, dim=-1).mean().item() * z.shape[0]
            n  += z.shape[0]
    return tl/n, tc/n, te/n


for epoch in range(N_EPOCHS):
    f_theta.train()
    tl, n = 0., 0
    for z, c in train_loader:
        z, c = z.to(DEVICE), c.to(DEVICE)
        optimizer.zero_grad()
        loss = cosine_loss(f_theta(z), c)
        loss.backward()
        optimizer.step()
        tl += loss.item() * z.shape[0]; n += z.shape[0]

    train_loss = tl / n
    val_loss, val_cos, val_euc = eval_epoch(val_loader)
    scheduler.step(val_euc)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_cos'].append(val_cos)
    history['val_euc'].append(val_euc)

    if val_euc < best_euc:
        best_euc, best_epoch, patience_counter = val_euc, epoch + 1, 0
        torch.save(f_theta.state_dict(), 'llava10_ftheta_best.pt')
    else:
        patience_counter += 1

    if (epoch + 1) % 5 == 0:
        print(f'Epoch {epoch+1:3d}/{N_EPOCHS} | train={train_loss:.4f} | '
              f'val_cos_loss={val_loss:.4f} | cos={val_cos:.4f} | '
              f'euc={val_euc:.4f} | patience={patience_counter}/{PATIENCE}')

    if patience_counter >= PATIENCE:
        print(f'Early stopping à l\'epoch {epoch + 1}')
        break

print(f'Meilleur modèle : epoch {best_epoch}, val_euc={best_euc:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('LLaVA-1.0 — Projection CLS MAE → CLS CLIP ViT-L/14 @ 224px')

axes[0].plot(history['train_loss'], label='train')
axes[0].plot(history['val_loss'],   label='val')
axes[0].axvline(best_epoch - 1, color='red', linestyle='--', alpha=0.5, label=f'best={best_epoch}')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss cosinus'); axes[0].set_title('Loss'); axes[0].legend()

axes[1].plot(history['val_cos'], color='darkorange')
axes[1].axvline(best_epoch - 1, color='red', linestyle='--', alpha=0.5)
axes[1].axhline(1.0, color='gray', linestyle='--', alpha=0.5)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Cos. sim.'); axes[1].set_title('Alignement val')

axes[2].plot(history['val_euc'], color='steelblue')
axes[2].axvline(best_epoch - 1, color='red', linestyle='--', alpha=0.5)
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('Distance euclidienne'); axes[2].set_title('Distance val')

plt.tight_layout()
plt.savefig('llava10_training_curves.png', dpi=150)
plt.show()